# **"1. Importing Libraries & Modules"**


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.metrics import accuracy_score
import yfinance as yf
from keras.models import Sequential
from keras.layers import LSTM, Dense

# **"2. Data Collecting"**

In [ ]:
stock = 'AAPL'  # Apple Stock Ticker
start_date = '2015-01-01'
end_date = '2024-01-01'
df = yf.download(stock, start=start_date, end=end_date)

# **"3. Data Preprocessing"**






# > 3.1 (Removing other parameters except Close price)



In [ ]:
data = df['Close'].values.reshape(-1, 1)
scaler = MinMaxScaler(feature_range=(0, 1))
data_scaled = scaler.fit_transform(data)



# > 3.2 (Splitting data into 80% = Training and 20% = Test)



In [ ]:
train_size = int(len(data_scaled) * 0.8)
train_data, test_data = data_scaled[:train_size], data_scaled[train_size:]



# > 3.3 (function to create sequence (loop/epochs) for LSTM)



In [ ]:
def create_dataset(data, time_step=100):
    X, Y = [], []
    for i in range(len(data) - time_step - 1):
        X.append(data[i:(i + time_step), 0])
        Y.append(data[i + time_step, 0])
    return np.array(X), np.array(Y)

time_step = 100
X_train, Y_train = create_dataset(train_data, time_step)
X_test, Y_test = create_dataset(test_data, time_step)



# > 3.4 (Reshaping for LSTM)



In [ ]:
X_train = X_train.reshape(X_train.shape[0], X_train.shape[1], 1)
X_test = X_test.reshape(X_test.shape[0], X_test.shape[1], 1)


# **"4. Building LSTM Model with MSE"**




In [ ]:
# while training we are using LSTM model in both the only difference is
# one has loss function = MSE and another has loss function = MAE
lstm_model = Sequential()
lstm_model.add(LSTM(units=50, return_sequences=True, input_shape=(time_step, 1)))
lstm_model.add(LSTM(units=50, return_sequences=False))
lstm_model.add(Dense(units=25, activation='relu'))
lstm_model.add(Dense(units=1))
#using the loss = MSE
lstm_model.compile(optimizer='adam', loss='mean_squared_error')

# **"5. Training MSE Model"**




In [ ]:
lstm_model.fit(X_train, Y_train, epochs=50, batch_size=32, validation_data=(X_test, Y_test), verbose=1)

# **"6. Building MAE Model"**




In [ ]:
mae_model = Sequential()
mae_model.add(LSTM(units=50, return_sequences=True, input_shape=(time_step, 1)))
mae_model.add(LSTM(units=50, return_sequences=False))
mae_model.add(Dense(units=25, activation='relu'))
mae_model.add(Dense(units=1))
# using the loss = MAE
mae_model.compile(optimizer='adam', loss='mean_absolute_error')

# **"7. Training MAE Model"**




In [ ]:
mae_model.fit(X_train, Y_train, epochs=50, batch_size=32, validation_data=(X_test, Y_test), verbose=1)

In [ ]:
# Saving the trained LSTM model in .keras format
lstm_model.save('lstm_model.keras')

# Saving the trained MAE model in .keras format
mae_model.save('mae_model.keras')

# Optionally, save the scaler (if needed later for preprocessing)
import pickle
with open('scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)


# **"8. Making the predictions"**




In [ ]:
# lstm model prediction
mse_predictions = mse_model.predict(X_test)
mse_predictions = scaler.inverse_transform(mse_predictions.reshape(-1, 1))
# mae model prediction
mae_predictions = mae_model.predict(X_test)
mae_predictions = scaler.inverse_transform(mae_predictions.reshape(-1, 1))

Y_test_actual = scaler.inverse_transform(Y_test.reshape(-1, 1))

# **"9. Visualizing (Comparisons)"**




In [ ]:
# Visualization (Comparisons)
plt.figure(figsize=(40, 20))



# > 9.1 (MSE) Prediction vs Actual Price)



In [ ]:
plt.plot(range(train_size + time_step, train_size + time_step + len(Y_test_actual)),
         Y_test_actual, color='green', label='Actual Stock Price')
plt.plot(range(train_size + time_step, train_size + time_step + len(mse_predictions)),
         mse_predictions, color='red', label='LSTM Predicted Price')
plt.title(f'{stock} - LSTM Prediction vs Original')
plt.xlabel('Time')
plt.ylabel('Stock Price')
plt.legend()



# > 9.2 (MAE Model Prediction vs Actual Price)



In [ ]:
# MAE Model Prediction vs Actual Price
plt.plot(range(train_size + time_step, train_size + time_step + len(Y_test_actual)),
         Y_test_actual, color='green', label='Actual Stock Price')
plt.plot(range(train_size + time_step, train_size + time_step + len(mae_predictions)),
         mae_predictions, color='brown', label='MAE Predicted Price')
plt.title(f'{stock} - MAE Model Prediction vs Original')
plt.xlabel('Time')
plt.ylabel('Stock Price')
plt.legend()



# > 9.3 (MSE Model Prediction vs MAE Model Prediction)



In [ ]:
# 3. LSTM vs MAE Model Prediction
plt.plot(range(train_size + time_step, train_size + time_step + len(mse_predictions)),
         mse_predictions, color='red', label='LSTM Predicted Price')
plt.plot(range(train_size + time_step, train_size + time_step + len(mae_predictions)),
         mae_predictions, color='brown', label='MAE Predicted Price')
plt.title(f'{stock} - LSTM vs MAE Model Prediction')
plt.xlabel('Time')
plt.ylabel('Stock Price')
plt.legend()

plt.tight_layout()
plt.show()

# **"10. Evaluating Metrics "**




In [ ]:
mse_mae = mean_absolute_error(Y_test_actual, mse_predictions)
mse_rmse = np.sqrt(mean_squared_error(Y_test_actual, mse_predictions))
mse_mape = np.mean(np.abs((Y_test_actual - mse_predictions) / Y_test_actual)) * 100

mae_mae = mean_absolute_error(Y_test_actual, mae_predictions)
mae_rmse = np.sqrt(mean_squared_error(Y_test_actual, mae_predictions))
mae_mape = np.mean(np.abs((Y_test_actual - mae_predictions) / Y_test_actual)) * 100

print(f"LSTM Model - MAE: {mse_mae}, RMSE: {mse_rmse}, MAPE: {mse_mape}%")
print(f"MAE Model - MAE: {mae_mae}, RMSE: {mae_rmse}, MAPE: {mae_mape}%")

# **"11. Accuracy Comparision"**




In [ ]:
from sklearn.metrics import r2_score, explained_variance_score

mse_r2 = r2_score(Y_test_actual, mse_predictions)
mae_r2 = r2_score(Y_test_actual, mae_predictions)

mse_evs = explained_variance_score(Y_test_actual, mse_predictions)
mae_evs = explained_variance_score(Y_test_actual, mae_predictions)

print(f"MSE Model R-squared: {mse_r2:.2f}")
print(f"MSE Model Explained Variance Score: {mse_evs:.2f}")

print(f"MAE Model R-squared: {mae_r2:.2f}")
print(f"MAE Model Explained Variance Score: {mae_evs:.2f}")